In [38]:
import os

from pyspark.sql import SparkSession

In [39]:
JAR_PATH = "/opt/spark/notebooks/clickhouse-jdbc-0.6.5-shaded.jar"

spark = SparkSession.builder \
    .appName("explore_readmission_spark_app") \
    .config("spark.jars", JAR_PATH) \
    .config("spark.driver.extraClassPath", JAR_PATH) \
    .config("spark.executor.extraClassPath", JAR_PATH) \
    .getOrCreate()


spark.sparkContext.setLogLevel("ERROR")

In [40]:
jdbc_url = f"jdbc:clickhouse://clickhouse:8123/{os.getenv('CLICKHOUSE_DB', 'default')}"

jdbc_options = {
    "url": jdbc_url,
    "user": os.getenv("CLICKHOUSE_USER", "default"),
    "password": os.getenv("CLICKHOUSE_PASSWORD", ""),
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
}

In [41]:
query = """
SELECT
    patient_id,
    encounter_id,
    toString(start_at) AS start_at,
    toString(stop_at) AS stop_at,
    age,
    gender,
    race,
    marital_status,
    total_encounters_before,
    inpatient_count_before,
    emergency_count_before,
    avg_duration_before,
    total_conditions_before,
    chronic_conditions_before,
    distinct_conditions_codes_before,
    readmitted_30d
FROM ehr.features_readmission
"""


df = (
    spark.read.format("jdbc")
    .options(**jdbc_options)
    .option("dbtable", f"({query}) AS t")
    .load()
)

In [42]:
df.printSchema()

root
 |-- patient_id: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- start_at: string (nullable = true)
 |-- stop_at: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- race: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- total_encounters_before: integer (nullable = true)
 |-- inpatient_count_before: integer (nullable = true)
 |-- emergency_count_before: integer (nullable = true)
 |-- avg_duration_before: float (nullable = true)
 |-- total_conditions_before: integer (nullable = true)
 |-- chronic_conditions_before: integer (nullable = true)
 |-- distinct_conditions_codes_before: integer (nullable = true)
 |-- readmitted_30d: integer (nullable = true)



In [43]:
print(f"Rows: {df.count()}")
df.show(5)

Rows: 5495
+--------------------+--------------------+-------------------+-------------------+---+------+-----+--------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+--------------+
|          patient_id|        encounter_id|           start_at|            stop_at|age|gender| race|marital_status|total_encounters_before|inpatient_count_before|emergency_count_before|avg_duration_before|total_conditions_before|chronic_conditions_before|distinct_conditions_codes_before|readmitted_30d|
+--------------------+--------------------+-------------------+-------------------+---+------+-----+--------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+--------------+
|03670b6a-8ff5-39e...|03670b6a-8ff5-39e...|1970-01-01 00:00:00|1970-01-01 00:

In [44]:
df.groupBy("readmitted_30d").count().show()

+--------------+-----+
|readmitted_30d|count|
+--------------+-----+
|             1|  940|
|             0| 4555|
+--------------+-----+



In [45]:
numeric_columns = [
    "age", "total_encounters_before", "inpatient_count_before",
    "emergency_count_before", "avg_duration_before",
    "total_conditions_before", "chronic_conditions_before",
    "distinct_conditions_codes_before"
]

df.describe(numeric_columns).show()

+-------+------------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+
|summary|               age|total_encounters_before|inpatient_count_before|emergency_count_before|avg_duration_before|total_conditions_before|chronic_conditions_before|distinct_conditions_codes_before|
+-------+------------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+
|  count|              5495|                   5495|                  5495|                  5495|               5495|                   5495|                     5495|                            5495|
|   mean|56.058052775250225|      74.21747042766151|    7.4787989080982715|     3.164331210191083|                NaN|     38.013648771610555|       22.593994540491355|              21.7878070

In [46]:
from pyspark.sql.functions import col, isnan, when

df.filter(isnan(col("avg_duration_before"))).count()

74

In [47]:
df = df.withColumn(
    "avg_duration_before",
    when(isnan(col("avg_duration_before")) | col("avg_duration_before").isNull(), 0.0)
    .otherwise(col("avg_duration_before"))
)

In [48]:
df.filter(isnan(col("avg_duration_before"))).count()

0

In [49]:
df.describe("avg_duration_before").show()

+-------+-------------------+
|summary|avg_duration_before|
+-------+-------------------+
|  count|               5495|
|   mean| 1084.1059673084358|
| stddev|  1513.118106133107|
|    min|                0.0|
|    max|     8711.599609375|
+-------+-------------------+



In [50]:
for numeric_col in numeric_columns:
    nan_count = df.filter(isnan(col(numeric_col))).count()
    null_count = df.filter(col(numeric_col).isNull()).count()
    print(f"{numeric_col}: NaN={nan_count}, NULL={null_count}")

age: NaN=0, NULL=0
total_encounters_before: NaN=0, NULL=0
inpatient_count_before: NaN=0, NULL=0
emergency_count_before: NaN=0, NULL=0
avg_duration_before: NaN=0, NULL=0
total_conditions_before: NaN=0, NULL=0
chronic_conditions_before: NaN=0, NULL=0
distinct_conditions_codes_before: NaN=0, NULL=0


In [51]:
df.describe(numeric_columns).show()

+-------+------------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+
|summary|               age|total_encounters_before|inpatient_count_before|emergency_count_before|avg_duration_before|total_conditions_before|chronic_conditions_before|distinct_conditions_codes_before|
+-------+------------------+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+
|  count|              5495|                   5495|                  5495|                  5495|               5495|                   5495|                     5495|                            5495|
|   mean|56.058052775250225|      74.21747042766151|    7.4787989080982715|     3.164331210191083| 1084.1059673084358|     38.013648771610555|       22.593994540491355|              21.7878070

In [52]:
patient_ids = df.select(col("patient_id")).distinct()

train_ids, test_ids = patient_ids.randomSplit([0.6, 0.4], seed=42)

In [53]:
train_df = df.join(train_ids, on="patient_id", how="inner")
test_df = df.join(test_ids, on="patient_id", how="inner")

In [54]:
print(f"Rows (train): {train_df.count()}, rows (test): {test_df.count()}")

Rows (train): 3396, rows (test): 2099


In [55]:
train_df = train_df.withColumn(
    "gender_encoded",
    when(col("gender") == "M", 1).otherwise(0)
)
test_df = test_df.withColumn(
    "gender_encoded",
    when(col("gender") == "M", 1).otherwise(0)
)

In [56]:
train_df.groupBy(col("gender"), col("gender_encoded")).count().show()

+------+--------------+-----+
|gender|gender_encoded|count|
+------+--------------+-----+
|     M|             1| 2108|
|     F|             0| 1288|
+------+--------------+-----+



In [57]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

indexer = StringIndexer(inputCol="marital_status", outputCol="marital_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="marital_idx", outputCol="marital_onehot")

indexer_model = indexer.fit(train_df)
train_df = indexer_model.transform(train_df)
test_df = indexer_model.transform(test_df)

encoder_model = encoder.fit(train_df)
train_df = encoder_model.transform(train_df)
test_df = encoder_model.transform(test_df)

In [58]:
train_df.select("gender", "gender_encoded", "marital_status", "marital_onehot").show(3, truncate=False)

+------+--------------+--------------+--------------+
|gender|gender_encoded|marital_status|marital_onehot|
+------+--------------+--------------+--------------+
|F     |0             |S             |(4,[1],[1.0]) |
|F     |0             |S             |(4,[1],[1.0]) |
|F     |0             |M             |(4,[0],[1.0]) |
+------+--------------+--------------+--------------+
only showing top 3 rows



In [59]:
feature_cols = [
    "age",
    "total_encounters_before",
    "inpatient_count_before",
    "emergency_count_before",
    "avg_duration_before",
    "total_conditions_before",
    "chronic_conditions_before",
    "distinct_conditions_codes_before",
    "gender_encoded",
    "marital_onehot",
]
train_df.select(*feature_cols).show(3, truncate=False)

+---+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+--------------+--------------+
|age|total_encounters_before|inpatient_count_before|emergency_count_before|avg_duration_before|total_conditions_before|chronic_conditions_before|distinct_conditions_codes_before|gender_encoded|marital_onehot|
+---+-----------------------+----------------------+----------------------+-------------------+-----------------------+-------------------------+--------------------------------+--------------+--------------+
|42 |3                      |0                     |0                     |0.0                |5                      |5                        |5                               |0             |(4,[1],[1.0]) |
|58 |8                      |1                     |1                     |13.875             |7                      |7                        |7                  

In [60]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_df = assembler.transform(train_df)
test_df = assembler.transform(test_df)

In [62]:
print(f"Train rows: {train_df.count()}")
train_df.select("features", "readmitted_30d").show(3, truncate=False)

Train rows: 3396
+-------------------------------------------------------------------------+--------------+
|features                                                                 |readmitted_30d|
+-------------------------------------------------------------------------+--------------+
|[46.0,34.0,1.0,3.0,1073.7059326171875,30.0,19.0,19.0,1.0,1.0,0.0,0.0,0.0]|0             |
|[41.0,13.0,0.0,2.0,42.07692337036133,11.0,8.0,10.0,1.0,1.0,0.0,0.0,0.0]  |0             |
|[65.0,72.0,1.0,3.0,190.5,72.0,47.0,40.0,1.0,1.0,0.0,0.0,0.0]             |0             |
+-------------------------------------------------------------------------+--------------+
only showing top 3 rows



In [63]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train_df)
train_df = scaler_model.transform(train_df)
test_df = scaler_model.transform(test_df)

In [65]:
print(f"Train rows: {train_df.count()}")
train_df.select("features_scaled", "readmitted_30d").show(3, truncate=False)

Train rows: 3396
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|features_scaled                                                                                                                                                                                                                                                          |readmitted_30d|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|[-0.5365088072688388,-0.36285385315358376,-0.5333639854644384,-0.032146110777559876,-0.027646946915851157,-0.17988811821072453,-0.238

In [69]:
from pyspark.ml.classification import LogisticRegression

train_df = (
    train_df
        .withColumn("weight",
                    when(col("readmitted_30d") == 1, 5.0).otherwise(1.0))
)

test_df = (
    test_df
        .withColumn("weight",
                    when(col("readmitted_30d") == 1, 5.0).otherwise(1.0))
)

lr = LogisticRegression(
    featuresCol="features_scaled",
    labelCol="readmitted_30d",
    weightCol="weight",
    maxIter=100
)

lr_model = lr.fit(train_df)

In [80]:
coefs = lr_model.coefficients.toArray()

names = [
    "age",
    "total_encounters_before",
    "inpatient_count_before",
    "emergency_count_before",
    "avg_duration_before",
    "total_conditions_before",
    "chronic_conditions_before",
    "distinct_conditions_codes_before",
    "gender_encoded",
    "marital_0",
    "marital_1",
    "marital_2",
    "marital_3",
]

for name, coef in sorted(zip(names, coefs), key=lambda x: abs(x[1]), reverse=True):
    print(f"{name:35s} {coef:+.4f}")

avg_duration_before                 +1.1956
emergency_count_before              -0.9310
distinct_conditions_codes_before    +0.8480
gender_encoded                      +0.2715
marital_2                           -0.1861
age                                 +0.1815
inpatient_count_before              -0.1698
total_conditions_before             -0.1571
marital_3                           -0.1443
marital_0                           +0.1217
chronic_conditions_before           -0.0881
total_encounters_before             +0.0632
marital_1                           +0.0550


In [85]:
predictions = lr_model.transform(test_df)